# 📊 Pipeline Preprocessing Job Data — versi diperbaiki (v3)

Notebook ini memperbaiki ekstraksi requirement dari dataset hasil scraping JobStreet.

## Apa yang diperbaiki dibanding versi sebelumnya
1. **Matching skill jauh lebih cepat** — diganti dari loop 137rb keyword (`re.search` per skill, ~350 ms/baris) ke **Aho-Corasick automaton** (~2 ms/baris). ±160 detik → ±1 detik untuk 448 baris.
2. **False-positive hilang** — skill DB disaring dari *stopword* (kata umum seperti `best`, `new`, `in`, `data`, `it`) dan entri ≤3 huruf yang ambigu. Sebelumnya 1 lowongan bisa menghasilkan ~98 "skill" yang banyak sampahnya; sekarang ~34 skill yang relevan.
3. **Hapus tumpang-tindih frasa** — kalau `Machine Learning` terdeteksi, `Learning` tidak ikut dimasukkan lagi. Begitu juga `Gradient Boosting` vs `Gradient`/`Boosting`.
4. **Batas kata (word boundary) benar** — `r` tidak lagi cocok di tengah kata `research`; hanya match sebagai token utuh.
5. **API key tidak di-hardcode** — diambil dari *environment variable* `GROQ_API_KEY` (lebih aman). Key lama Anda sebaiknya di-*revoke* karena sudah pernah tampil di file.
6. **Path relatif & portable** — tidak lagi hard-coded ke `D:/ProjekAkhir/...`.

> Catatan: kalau ingin lebih agresif lagi membersihkan, tambahkan kata ke `EXTRA_STOPWORDS`.


### 1. Import library & konfigurasi

In [40]:
import os
import re
import json
import time
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# Aho-Corasick untuk pencarian multi-keyword super cepat
# install sekali: pip install pyahocorasick
import ahocorasick

# ==========================================================
# KONFIGURASI PATH (ubah sesuai struktur folder Anda)
# ==========================================================
RAW_CSV       = "../dataset/main_data_llm/job_portal/raw/dataset_web-scraper_2026-06-02_02-56-16-796.csv"   # dataset hasil scraping
SKILLS_CSV    = "skills_cleaned_v2.csv"                                 # database skill
OUTPUT_JSON   = "dataset_job_processed_final.json"
OUTPUT_CSV    = "dataset_job_processed_final.csv"

# Mode sampel untuk uji cepat (set False untuk proses semua data)
SAMPLE_MODE = False
SAMPLE_SIZE = 50


### 2. Load database skill + penyaringan stopword

Langkah ini yang paling menentukan kualitas. Kita buang kata-kata umum yang bikin false-positive, lalu bangun automaton Aho-Corasick sekali saja.

In [41]:
# ==========================================================
# STOPWORD: kata umum (EN + ID) yang TIDAK boleh dianggap skill
# Tambahkan kata di sini kalau masih ada noise yang lolos.
# ==========================================================
EXTRA_STOPWORDS = set("""
a an the of to in on at by for and or but with from into as is are was were be been being this that
these those it its they them their we our you your he she his her i me my new best most high low good
great clear strong solid based able will can may any all some more less very much many few each every
other another such same own only just also not no yes do does did done make made get got go going work
working works job jobs role roles team teams data business company companies client clients project
projects product products related things experience experiences knowledge skill skills ability abilities
requirement requirements scale responsibility responsibilities professional environment environments
method methods test tools operation operations problem problems solution solutions challenge challenges
insight insights join us why how what when where which who whom plus etc using use used user users travel
year years month months day days time times level levels field fields area areas part parts include
including includes provide provided provides ensure ensures ensuring support supports proven track record
decision making leadership communication research stack practice expertise relationships verbal written
technical strategic mentor benefits proficient detection regression apply identify understand utilize
results input report reports control standard target program material medical department
dan atau yang untuk dengan pada dari ke di se para dalam akan dapat juga tidak ya kerja pekerjaan
perusahaan pengalaman kemampuan keahlian tanggung jawab lingkungan minimal
""".split())

# Singkatan teknis pendek yang TETAP ingin dipertahankan walau pendek
WHITELIST = set(['r','c','go','ml','ai','bi','qa','ux','ui','db','js','ci','cd','aws','gcp',
                 'sql','etl','sap','erp','crm','php','css','api','xml','jvm','tdd','sdk','npm'])
COMMON_ACRONYMS = set(['c#','c++','sql','api','xml','aws','gcp','ui','ux','ml','ai','js','php','r','db','ci','cd','erp','crm','sap','net'])
SKILL_LABELS = {}

BAD_SKILL_PATTERN = re.compile(
    r'\b(?:ability to|ability|experience in|experience|experienced|knowledge of|knowledge in|knowledge|requirement|requirements|responsibility|responsibilities|skill|skills|competency|competencies|role|roles|job|jobs|work|training|practice|professional|background|support|including|related|etc|company|companies|department|departments)\b',
    flags=re.I,
)


def normalize_skill_keyword(skill: str) -> str:
    s = str(skill).strip().lower()
    s = re.sub(r'[\u2018\u2019\u201c\u201d"]', '', s)
    s = re.sub(r'\(.*?\)', ' ', s)
    s = re.sub(r'\b(?:years?|yrs?|months?|mos?|weeks?|days?|hours?|experience|experienced|v\d+|version)\b', ' ', s)
    s = re.sub(r'[^a-z0-9#+/]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def format_skill_label(normalized: str) -> str:
    if not normalized:
        return normalized
    if normalized.upper() == normalized:
        return normalized
    tokens = []
    for token in normalized.split():
        if token in COMMON_ACRONYMS:
            tokens.append(token.upper())
        elif '/' in token:
            tokens.append('/'.join(
                t.upper() if t in COMMON_ACRONYMS else t.title()
                for t in token.split('/') if t
            ))
        else:
            tokens.append(token.title())
    return ' '.join(tokens)


def is_useful_skill(raw: str, normalized: str) -> bool:
    if normalized in WHITELIST:
        return True
    if raw in EXTRA_STOPWORDS or normalized in EXTRA_STOPWORDS:
        return False
    if not normalized or len(normalized) <= 2:
        return False
    if len(normalized) == 3 and normalized.isalpha():
        return False
    if BAD_SKILL_PATTERN.search(raw):
        return False
    if re.search(r'^[^a-z0-9]*$', normalized):
        return False
    return True


def load_skill_keywords(path: str) -> set:
    if not os.path.exists(path):
        print(f"⚠️  File skill tidak ditemukan: {path}")
        return set()
    try:
        df_skill = pd.read_csv(path, encoding='utf-8', on_bad_lines='skip')
    except UnicodeDecodeError:
        df_skill = pd.read_csv(path, encoding='latin1', on_bad_lines='skip')
    col = df_skill.columns[0]
    raw = df_skill[col].dropna().astype(str).tolist()
    raw_n = len(raw)
    canonical = {}
    for item in raw:
        normalized = normalize_skill_keyword(item)
        if not is_useful_skill(item, normalized):
            continue
        label = format_skill_label(normalized)
        if normalized not in canonical or len(label) < len(canonical[normalized]):
            canonical[normalized] = label
    SKILL_LABELS.clear()
    SKILL_LABELS.update(canonical)
    pd.DataFrame(list(canonical.items())).to_csv("skills_cleaned_v3.csv", header=False, index=False)
    print(f"✅ Skill DB: {raw_n} -> {len(canonical)} dipakai (dibuang {raw_n - len(canonical)} entri noise)")
    return set(canonical.keys())


SKILL_KEYWORDS = load_skill_keywords(SKILLS_CSV)

# Bangun automaton Aho-Corasick (sekali saja, dipakai untuk semua baris)
def build_automaton(keywords: set):
    A = ahocorasick.Automaton()
    for kw in keywords:
        A.add_word(kw, kw)
    A.make_automaton()
    return A

SKILL_AUTOMATON = build_automaton(SKILL_KEYWORDS)
print("✅ Automaton siap.")


✅ Skill DB: 124029 -> 124027 dipakai (dibuang 2 entri noise)
✅ Automaton siap.


### 3. Fungsi cleaning HTML & ekstraksi skill (cepat + bersih)

In [42]:
def clean_html(html_text):
    """Ambil teks bersih dari HTML deskripsi lowongan (lowercase)."""
    if pd.isna(html_text):
        return ""
    return BeautifulSoup(html_text, "html.parser").get_text(separator=" ").lower()


def _is_word_boundary(text, start, end):
    """Pastikan match adalah token utuh, bukan bagian dari kata lain."""
    before = text[start - 1] if start > 0 else " "
    after  = text[end + 1] if end + 1 < len(text) else " "
    left_ok  = not (before.isalnum() or before == "_")
    right_ok = not (after.isalnum() or after == "_")
    return left_ok and right_ok


def _remove_overlaps(matches: set) -> set:
    """Buang skill yang merupakan bagian dari frasa skill lebih panjang.
    Contoh: kalau ada 'machine learning', maka 'learning' dibuang."""
    multi = [m for m in matches if " " in m]
    joined = " || ".join(multi)
    kept = set()
    for s in matches:
        if " " not in s and multi and re.search(r"\b" + re.escape(s) + r"\b", joined):
            continue
        kept.add(s)
    return kept


def normalize_match_text(text: str) -> str:
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[\u2018\u2019\u201c\u201d"]', '', text)
    text = re.sub(r'[^a-z0-9#+/]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def extract_skills(text: str) -> list:
    """Ekstrak skill dari seluruh deskripsi menggunakan Aho-Corasick."""
    if not text:
        return []
    text_norm = normalize_match_text(text)
    found = set()
    for end_idx, kw in SKILL_AUTOMATON.iter(text_norm):
        start_idx = end_idx - len(kw) + 1
        if _is_word_boundary(text_norm, start_idx, end_idx):
            found.add(kw)
    found = _remove_overlaps(found)
    pretty = {SKILL_LABELS.get(s, format_skill_label(s)) for s in found}
    return sorted(pretty)


def extract_requirements_text(text: str) -> str:
    """Versi string (kompatibel dengan kolom lama 'extracted_requirements')."""
    skills = extract_skills(text)
    exp = re.search(r"(\d+\+?)\s*(?:-\s*\d+)?\s*(?:years?|tahun)", text)
    if exp:
        skills.append(f"Exp: {exp.group(0).strip()}")
    return ", ".join(skills) if skills else "General Requirement"


### 4. Ekstraksi experience / GPA / degree (regex dulu, Groq sebagai fallback)

Regex sudah cukup untuk mayoritas kasus. Groq hanya dipanggil saat regex belum lengkap, sehingga hemat kuota API.

In [43]:
# Ambil API key dari environment, JANGAN hardcode di notebook.
#   set di terminal:  export GROQ_API_KEY="gsk_xxx"   (Linux/Mac)
#                     setx GROQ_API_KEY "gsk_xxx"      (Windows)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
USE_GROQ = bool(GROQ_API_KEY)

groq_client = None
if USE_GROQ:
    try:
        from groq import Groq
        groq_client = Groq(api_key=GROQ_API_KEY)
        print("✅ Groq aktif (fallback NLP tersedia).")
    except Exception as e:
        USE_GROQ = False
        print(f"⚠️  Groq tidak bisa diinisialisasi, lanjut regex-only: {e}")
else:
    print("ℹ️  GROQ_API_KEY tidak diset -> mode regex-only (tetap jalan, tanpa fallback NLP).")


def extract_min_experience_regex(text):
    if not text:
        return None
    t = text.lower()
    patterns = [
        r"(?:minimum|minimal|at least|require[d]?|setidaknya)\s+(?:of\s+)?(\d+)\+?\s*(?:to|-|through)?\s*\d*\s*years?\s+(?:of\s+)?(?:work|professional|relevant|kerja|pengalaman)",
        r"(?:experience|pengalaman)\s*[:=]\s*(\d+)\+?\s*(?:to|-|through)?\s*\d*\s*years?\b",
        r"(\d+)\+?\s*(?:to|-|through)?\s*\d*\s*years?\s+of\s+(?:professional|relevant|work|working|kerja|pengalaman)",
        r"(?:minimum|minimal|at least|setidaknya)?\s*(\d+)\+?\s*(?:to|-|through)?\s*\d*\s*years?\s+experience\b",
        r"(?:pengalaman|bekerja)\s+(?:minimal|minimum|setidaknya)?\s*:?\s*(\d+)\+?\s*(?:hingga|-|sampai)?\s*\d*\s*tahun",
        r"(\d+)\s*-\s*\d+\s*years?\s+(?:of\s+)?(?:experience|kerja|pengalaman)",
        r"(?:minimum|require|at least|prefer|setidaknya|minimal)\s+(?:of\s+)?(\d+)\+?\s*years?\b",
    ]
    yrs = []
    for p in patterns:
        for m in re.finditer(p, t):
            try:
                v = int(m.group(1))
                if 0 <= v <= 50:
                    yrs.append(v)
            except (ValueError, IndexError):
                pass
    return min(yrs) if yrs else None


def extract_min_gpa_regex(text):
    if not text:
        return None
    t = text.lower()
    for p in [r"(?:gpa|ipk)\s*[:\s]+([0-4][.,][0-9]{1,2})"]:
        m = re.search(p, t)
        if m:
            try:
                v = float(m.group(1).replace(",", "."))
                if 0 <= v <= 4.0:
                    return round(v, 2)
            except ValueError:
                pass
    return None


def extract_min_degree_regex(text):
    if not text:
        return None
    t = text.lower()
    levels = [
        (r"\b(?:phd|professor|profesor|doktor|s\.?3|s3)\b", "PhD"),
        (r"\b(?:master|s\.?2|s2|pascasarjana|postgraduate|magister)\b", "Master"),
        (r"\b(?:bachelor|s\.?1|s1|sarjana|strata\s*1|d\.?4|d4|sarjana\s*terapan)\b", "Bachelor"),
        (r"\b(?:diploma|d\.?3|d3|amd)\b", "Diploma"),
    ]
    for pat, name in levels:
        if re.search(pat, t):
            return name
    return None


def extract_education_requirements(text, retry=0, max_retries=2):
    result = {"min_experience_years": None, "min_gpa": None,
              "min_degree": None, "required_majors": [], "extraction_method": "regex"}
    if not text:
        return result

    exp = extract_min_experience_regex(text)
    gpa = extract_min_gpa_regex(text)
    deg = extract_min_degree_regex(text)
    result.update(min_experience_years=exp, min_gpa=gpa, min_degree=deg)

    found = sum(x is not None for x in (exp, gpa, deg))
    if found >= 3 or not USE_GROQ:
        return result

    # ---- Fallback Groq hanya bila regex belum lengkap ----
    try:
        system_prompt = (
            "Kamu sistem ATS expert. Analisis job description dan return JSON: "
            '{"min_experience_years": <angka atau null>, "min_gpa": <angka atau null>, '
            '"min_degree": "<PhD/Master/Bachelor/Diploma atau null>", '
            '"required_majors": ["<major1>", "<major2>"]}. '
            "D4=Bachelor, D3=Diploma. Hanya return JSON."
        )
        completion = groq_client.chat.completions.create(
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": f"Job Description:\n{text[:2000]}"}],
            model="llama-3.1-8b-instant",
            temperature=0.2,
            response_format={"type": "json_object"},
            timeout=15,
        )
        g = json.loads(completion.choices[0].message.content)
        if exp is None and g.get("min_experience_years"):
            result["min_experience_years"] = g["min_experience_years"]
        if gpa is None and g.get("min_gpa"):
            result["min_gpa"] = g["min_gpa"]
        if deg is None and g.get("min_degree"):
            result["min_degree"] = g["min_degree"]
        result["required_majors"] = g.get("required_majors", [])
        result["extraction_method"] = "regex+groq"
    except Exception as e:
        if retry < max_retries:
            time.sleep(1)
            return extract_education_requirements(text, retry + 1, max_retries)
        print(f"⚠️  Groq gagal (regex-only): {e}")
        result["extraction_method"] = "regex_only"
    return result


✅ Groq aktif (fallback NLP tersedia).


### 5. Load dataset, hapus duplikat, lalu jalankan ekstraksi

In [44]:
df = pd.read_csv(RAW_CSV)
print(f"Data dimuat: {len(df)} baris, {len(df.columns)} kolom")

# Hapus duplikat berdasarkan jobId
if "jobId" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["jobId"], keep="first").reset_index(drop=True)
    print(f"Duplikat dihapus: {before - len(df)}  ->  sisa {len(df)} baris")
else:
    print("⚠️  Kolom 'jobId' tidak ada, lewati dedup.")

if SAMPLE_MODE:
    df = df.head(SAMPLE_SIZE).copy()
    print(f"🧪 SAMPLE MODE: hanya {len(df)} baris diproses")


Data dimuat: 292 baris, 21 kolom
Duplikat dihapus: 146  ->  sisa 146 baris


In [45]:
# --- 5a. Bersihkan HTML + ekstrak skill (cepat) ---
t0 = time.time()
df["clean_description"] = df["jobDescriptionHtml"].apply(clean_html)
df["extracted_skills"] = df["clean_description"].apply(extract_skills)          # list
df["extracted_requirements"] = df["clean_description"].apply(extract_requirements_text)  # string
print(f"✅ Skill extraction selesai dalam {time.time()-t0:.1f}s "
      f"(rata-rata {df['extracted_skills'].apply(len).mean():.1f} skill/lowongan)")


✅ Skill extraction selesai dalam 1.3s (rata-rata 49.2 skill/lowongan)


In [46]:
# --- 5b. Ekstrak experience / GPA / degree / major ---
t0 = time.time()
records = []
n = len(df)
for i, row in enumerate(df.itertuples(index=False), 1):
    records.append(extract_education_requirements(getattr(row, "clean_description", "")))
    if i % max(1, n // 10) == 0:
        el = time.time() - t0
        print(f"  [{i}/{n}] {el:.1f}s, ~{el/i*(n-i):.1f}s tersisa")

edu = pd.DataFrame(records)
for c in ["min_experience_years", "min_gpa", "min_degree", "required_majors", "extraction_method"]:
    df[c] = edu[c].values
print(f"✅ Education extraction selesai dalam {time.time()-t0:.1f}s")


  [14/146] 9.0s, ~84.8s tersisa
  [28/146] 75.6s, ~318.4s tersisa
  [42/146] 141.5s, ~350.3s tersisa
  [56/146] 219.0s, ~351.9s tersisa
  [70/146] 287.2s, ~311.8s tersisa
  [84/146] 354.5s, ~261.7s tersisa
  [98/146] 423.7s, ~207.5s tersisa
  [112/146] 493.4s, ~149.8s tersisa
  [126/146] 561.5s, ~89.1s tersisa
  [140/146] 631.2s, ~27.1s tersisa
✅ Education extraction selesai dalam 653.0s


### 6. Susun output & simpan (JSON + CSV)

In [47]:
def build_url(jobid):
    try:
        # Pengecekan tambahan untuk memastikan jobId valid dan bukan "Unknown"
        if pd.isna(jobid) or jobid == "Unknown" or not str(jobid).strip():
            return ""
        return f"https://id.jobstreet.com/id/job/{int(float(jobid))}"
    except (ValueError, TypeError):
        return ""

# Langsung timpa (overwrite) isi URL menggunakan jobId
if "jobId" in df.columns:
    df["jobSearchUrl"] = df["jobId"].apply(build_url)

# PERBAIKAN: Ubah "sourceUrl" menjadi "job_url" agar cocok dengan skema SQLite Anda
rename_map = {
    "classifications/0/main": "category_main",
    "classifications/0/sub":  "category_sub",
    "company/description":    "company_description",
    "company/name":           "company_name",
    "workTypes/0":            "work_type",
    "jobSearchUrl":           "job_url",          # <-- Diperbaiki di sini
    "title":                  "job_title",
    "clean_description":      "description_text",
    "extracted_requirements": "requirements",
}

cols = list(rename_map.keys()) + [
    "salary", "extracted_skills", "min_experience_years",
    "min_gpa", "min_degree", "required_majors", "extraction_method",
]

# Tambahkan kolom kosong jika belum ada
for c in cols:
    if c not in df.columns:
        df[c] = None

out = df[cols].rename(columns=rename_map)

# Simpan JSON (utama)
data_dict = out.replace({np.nan: None}).to_dict(orient="records")
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data_dict, f, indent=4, ensure_ascii=False)

# Simpan CSV (cadangan; list di-join agar rapi di CSV)
out_csv = out.copy()

# Pengecekan aman sebelum melakukan join untuk list
if "extracted_skills" in out_csv.columns:
    out_csv["extracted_skills"] = out_csv["extracted_skills"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x)

if "required_majors" in out_csv.columns:
    out_csv["required_majors"] = out_csv["required_majors"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x)

out_csv.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Disimpan: {OUTPUT_JSON}  ({len(data_dict)} records)")
print(f"✅ Disimpan: {OUTPUT_CSV}")

✅ Disimpan: dataset_job_processed_final.json  (146 records)
✅ Disimpan: dataset_job_processed_final.csv


In [48]:
# --- Cek hasil cepat ---
print("Metode ekstraksi (distribusi):")
print(df["extraction_method"].value_counts(), "\n")
print("Contoh 1 record:")
print(json.dumps(data_dict[0], indent=2, ensure_ascii=False)[:1500])


Metode ekstraksi (distribusi):
extraction_method
regex+groq    145
regex           1
Name: count, dtype: int64 

Contoh 1 record:
{
  "category_main": "Teknologi Informasi",
  "category_sub": "Data Science",
  "company_description": null,
  "company_name": "PT Indocater",
  "work_type": "Kontrak/Temporer",
  "job_url": "https://id.jobstreet.com/id/job/92079700",
  "job_title": "Associate Data Planning Analyst",
  "description_text": "minimal pendidikan s1 dari jurusan: petroleum engineering, mining engineering, chemical engineering, mechanical engineering, electrical engineering, atau industrial engineering. memiliki pengalaman kerja 2–5 tahun di industri minyak dan gas. memiliki kemampuan bahasa inggris yang baik, baik secara lisan maupun tulisan. menguasai microsoft office (excel, powerpoint, outlook, dan word). memiliki kemampuan bahasa inggris untuk technical documentation dan agreement di industri oil & gas (joa).",
  "requirements": "Chemical, Electrical Engineering, Industrial, 